## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [4]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from mlflow.models import infer_signature
...

Ellipsis

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [5]:
# Implement transformation logic as in session 2
import joblib

df_input = pd.read_csv("../Session2_class/datasets/Churn_Modelling_train_test.csv")
df_input = df_input.dropna(axis=0)

def balance_dataset(df: pd.DataFrame, target: str) -> pd.DataFrame:
    df_y0 = df[df[target] == 0]
    df_y1 = df[df[target] == 1]
    
    # Undersampling the majority class
    min_size = len(df_y1)
    df_majority_downsampled = df_y0.sample(n=min_size, random_state=42)
    
    df = pd.concat([df_majority_downsampled, df_y1])
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced = balance_dataset(df_input, 'Exited')

class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = ['RowNumber', 'CustomerId', 'Surname']
        self.CATEGORICAL_COLUMNS = ['Geography', 'Gender']
        self.encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")

    def transform(self, df: pd.DataFrame, is_training: bool = False) -> pd.DataFrame:
        df = df.drop(columns=self.DROP_COLUMNS)
        
        if is_training:
           self.encoder.fit(df[self.CATEGORICAL_COLUMNS])
            
        encoded_features = self.encoder.transform(df[self.CATEGORICAL_COLUMNS])
        df = df.drop(columns=self.CATEGORICAL_COLUMNS)
        df = pd.concat([df, encoded_features], axis=1)
        return df
transformer = Transformer()
df_transformed = transformer.transform(df_balanced, is_training = True)
print(df_transformed.head())

   CreditScore   Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          674  45.0       7  142072.02              1        1.0   
1          438  54.0       2       0.00              1        0.0   
2          749  47.0       9  110022.74              1        0.0   
3          724  34.0       6  118235.70              2        0.0   
4          586  46.0       0       0.00              3        0.0   

   IsActiveMember  EstimatedSalary  Exited  Geography_Germany  \
0             0.0         37013.29       0                0.0   
1             0.0        191763.07       1                0.0   
2             1.0        135655.29       1                1.0   
3             0.0        157137.23       0                1.0   
4             1.0        131553.82       1                0.0   

   Geography_Spain  Gender_Male  
0              0.0          1.0  
1              1.0          0.0  
2              0.0          1.0  
3              0.0          1.0  
4              0.0      

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = df_transformed.drop('Exited', axis=1)
y = df_transformed['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "max_depth": 6,
    "min_samples_split": 10,
    "random_state": 21
}

dt_model = DecisionTreeClassifier(**params)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.7514
F1 Score: 0.7570


In [156]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

# Create a new MLflow Experiment
mlflow.set_experiment("Practice Experiment S3- Hemangini Jena")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session3 Assignment", "Bank Customer Churn ML Experiment")

    # Infer the model signature
    signature = infer_signature(X_train, dt_model.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=dt_model,
        artifact_path="decision_tree_bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="churn-model-dt")

c:\Users\sudhi\Downloads\MLOps\Session 1_ML\mlops-and-system-design\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'churn-model-dt' already exists. Creating a new version of this model...
2026/05/24 19:57:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model v

🏃 View run delightful-ram-618 at: http://127.0.0.1:8080/#/experiments/658896502001904195/runs/4f822077821f418083a65d833f2a00be
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/658896502001904195


Created version '9' of model 'churn-model-dt'.


In [11]:
import joblib

PATH = 'model_utils/'
joblib.dump(transformer.encoder,f'{PATH}encoder.pkl')
joblib.dump(dt_model, f'{PATH}dt_model.pkl')

['model_utils/dt_model.pkl']

In [6]:
model_uri='runs:/4f822077821f418083a65d833f2a00be/decision_tree_bank_model'

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [178]:
# import validation dataset to test inference
df_val = pd.read_csv("../Session3_class/val_datasets/Churn_Modelling_val.csv")
df_val = df_val.dropna(axis=0)
print(df_val['Geography'].unique())
print("Total True Nulls:", df_val['Geography'].isna().sum())

['Spain' 'France' 'Germany']
Total True Nulls: 0


In [179]:
y_validation = df_val["Exited"]
df_val = df_val.drop("Exited", axis = 1)

In [180]:
import requests
import json

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [181]:
# transform data - if necessary
import joblib

class ModelBatchPredictor:
    CATEGORICAL_COLUMNS = ['Geography', 'Gender']
        
    def __init__(self):
        self.encoder = joblib.load(f'{PATH}encoder.pkl')
        self.transformer = Transformer()
        self.transformer.encoder = self.encoder

    def _apply_transformation(self, df: pd.DataFrame):
        df = self.transformer.transform(df.copy())

        return df 

##### Batch Inference

In [12]:
# define a function to implement batch inference with mlflow
def batch_inference_with_mlflow(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    predictor = ModelBatchPredictor()
    input = predictor._apply_transformation(input)
    predictions = model.predict(input)
        
    return predictions
    
def batch_inference_without_mlflow(model_path: str, input: pd.DataFrame):
    model = joblib.load(model_path)
    predictor = ModelBatchPredictor()
    input = predictor._apply_transformation(input)
    predictions = model.predict(input)

    return predictions

In [183]:
# define the model uri that should be used
model_uri = 'runs:/4f822077821f418083a65d833f2a00be/decision_tree_bank_model'
input = df_val
batch_prediction_result = batch_inference_with_mlflow(model_uri, input)

In [184]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

print(batch_prediction_result[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_result))
print("----")
print(accuracy_score(y_validation, batch_prediction_result))

[0 0 0 1 0]
----
[[590 212]
 [ 39 159]]
----
0.749


In [185]:
batch_prediction_without_mlflow = batch_inference_without_mlflow(
    model_path = f'{PATH}dt_model.pkl',
    input = df_val
)

In [186]:
print(batch_prediction_without_mlflow[:5])
print("----")
print(confusion_matrix(y_validation, batch_prediction_without_mlflow))
print("----")
print(accuracy_score(y_validation, batch_prediction_without_mlflow))

[0 0 0 1 0]
----
[[590 212]
 [ 39 159]]
----
0.749


##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/25b753bcbe6942dd891ac1a1c2587821/decision_tree_bank_model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [17]:
import requests
import json

In [7]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv("../Session3_class/val_datasets/Churn_Modelling_val.csv").head(1)
print(df_validation)
print(df_validation["Exited"])
df_validation = df_validation.drop("Exited", axis = 1)

   RowNumber  CustomerId   Surname  CreditScore Geography Gender   Age  \
0       8607    15694581  Rawlings          807     Spain   Male  42.0   

   Tenure  Balance  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  \
0       5      0.0              2        1.0             1.0          74900.9   

   Exited  
0       0  
0    0
Name: Exited, dtype: int64


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [13]:
# transform data - if necessary
class MLflowInferenceOnline:
    ONE_HOT_ENCODE_COLUMNS= [
            "Geography",
            "Gender",
        ]
    def __init__(self, host="http://127.0.0.1", port=5000):
        self.url = f"{host}:{port}/invocations"
        self.encoder = joblib.load(f'{PATH}encoder.pkl')
        self.transformer = Transformer()
        self.transformer.encoder = self.encoder

    def _apply_transformation(self,df: pd.DataFrame):
        df = self.transformer.transform(df.copy())

        return df 

    def predict_csv(self, csv_data: str):
        """Send CSV-formatted request"""
        headers = {"Content-Type": "text/csv"}
        response = requests.post(self.url, headers=headers, data=csv_data)
        return response
    
    
ml_flow_online_inference_client = MLflowInferenceOnline()

In [16]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(pandas_data: pd.DataFrame):
        headers = {"Content-Type": "application/json"}
        pandas_data = ml_flow_online_inference_client._apply_transformation(pandas_data)
        payload = {"dataframe_split": pandas_data.to_dict(orient="split")}
        response = requests.post(ml_flow_online_inference_client.url, headers=headers, data=json.dumps(payload))
        return response


In [187]:
response_pandas = online_inference_pandas(df_validation)
response_pandas.content

b'{"predictions": [0]}'

In [1]:
json_data = {
    "dataframe_split": { 
    "columns": ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary", "Geography_Germany", "Geography_Spain", "Gender_Male"],
    "data": [[807, 42.0, 5, 0.0, 2, 1.0, 1.0, 74900.9, 0.0, 1.0, 1.0]]}}

In [14]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(json_data: dict):
    """Send JSON (pandas-style) request"""
    headers = {"Content-Type": "application/json"}
    response = requests.post(ml_flow_online_inference_client.url, headers=headers, json=json_data)
    return response

In [18]:
response_json = online_inference_json(json_data)
response_json.content

b'{"predictions": [0]}'